# 06 — Scenario Generation

Generate future scenarios using CIB-based clustering:
1. **Cluster** drivers via hierarchical clustering on CIB interaction profiles
2. **LLM archetype generation** from cluster data + influence ranking + tension pairs
3. **CIB consistency check** validates state combinations
4. **Scenario narratives** grounded in RAG + CIB context

In [1]:
import sys, os, json
import numpy as np
sys.path.insert(0, os.path.abspath(".."))

from src.config import CHROMA_PERSIST_DIR
from src.llm import safe_chat_json, embed
from src.traceability import TechDriver, Scenario, DriverAssumption, ScenarioType, DriverOrigin
from src import prompts

import chromadb
client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
collection = client.get_collection("knowledge_base")

with open("../data/outputs/merge_state.json") as f:
    drivers = [TechDriver(**d) for d in json.load(f)["unified_drivers"]]

with open("../data/outputs/cib_state.json") as f:
    cib = json.load(f)

cib_matrix = np.array(cib["matrix"])
cib_driver_ids = cib["driver_ids"]
cib_id_to_idx = {did: i for i, did in enumerate(cib_driver_ids)}
driver_by_id = {d.id: d for d in drivers}

print(f"Drivers: {len(drivers)}")
print(f"CIB matrix: {cib_matrix.shape}")
print(f"CIB influence ranking:")
for did, score in sorted(cib["influence"].items(), key=lambda x: x[1], reverse=True)[:5]:
    d = driver_by_id.get(did)
    if d:
        print(f"  {score:+3d} | {d.name}")

Drivers: 14
CIB matrix: (14, 14)
CIB influence ranking:
  +11 | Ultra Wideband RF Frontend and Millimeter-Wave/Sub-THz Spectrum Monitoring
  +10 | Real-Time Digitizer and Photonic Signal Processing for Ultra-Wideband Monitoring
   +9 | Cross-Border and Harmonized Spectrum Monitoring Coordination
   +7 | I/Q Data Streaming Interface and Edge Computing for Distributed Spectrum Monitoring
   +7 | Digital Twin Networks for Spectrum Monitoring


In [2]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Step 1: Hierarchical clustering on CIB matrix rows
n_cib = cib_matrix.shape[0]
N_CLUSTERS = 4

dist_matrix = np.zeros((n_cib, n_cib))
for i in range(n_cib):
    for j in range(n_cib):
        dist_matrix[i][j] = np.linalg.norm(cib_matrix[i] - cib_matrix[j])

condensed = squareform(dist_matrix)
Z = linkage(condensed, method='ward')
cluster_labels = fcluster(Z, t=N_CLUSTERS, criterion='maxclust')

clusters: dict[int, list[int]] = {}
for idx, label in enumerate(cluster_labels):
    clusters.setdefault(int(label), []).append(idx)

driver_by_id = {d.id: d for d in drivers}
cib_drivers_list = [driver_by_id.get(did) for did in cib_driver_ids]

print(f"=== CIB-Based Driver Clusters ({N_CLUSTERS} clusters) ===\n")
cluster_descriptions = []
for cid, indices in sorted(clusters.items()):
    names = [cib_drivers_list[i].name[:50] for i in indices if cib_drivers_list[i]]
    avg_influence = np.mean([cib["influence"].get(cib_driver_ids[i], 0) for i in indices])
    avg_dependence = np.mean([cib["dependence"].get(cib_driver_ids[i], 0) for i in indices])
    print(f"Cluster {cid} (avg influence: {avg_influence:+.1f}, avg dependence: {avg_dependence:+.1f}):")
    for name in names:
        print(f"  • {name}")
    cluster_descriptions.append(
        f"Cluster {cid} (influence={avg_influence:+.1f}, dependence={avg_dependence:+.1f}): {', '.join(names)}"
    )
    print()

# Step 2: Identify tensions (pairs with score <= -1)
tension_pairs = []
for i in range(n_cib):
    for j in range(n_cib):
        if i != j and cib_matrix[i][j] <= -1:
            tension_pairs.append(
                f"{cib_drivers_list[i].name[:40]} → {cib_drivers_list[j].name[:40]}: {cib_matrix[i][j]}"
            )

influence_ranking = sorted(
    [(cib_drivers_list[i].name, cib["influence"].get(cib_driver_ids[i], 0)) for i in range(n_cib)],
    key=lambda x: x[1], reverse=True
)
influence_text = "\n".join([f"  {name[:50]}: influence={score:+d}" for name, score in influence_ranking])

print(f"Tensions (score <= -1): {len(tension_pairs)}")
for tp in tension_pairs[:15]:
    print(f"  {tp}")
if len(tension_pairs) > 15:
    print(f"  ... and {len(tension_pairs) - 15} more")

# Step 3: LLM proposes 6 scenario archetypes with enforced typology
archetype_prompt = f"""Based on this CIB analysis of {n_cib} technology drivers for regulatory frequency monitoring:

DRIVER CLUSTERS (from cross-impact analysis):
{chr(10).join(cluster_descriptions)}

INFLUENCE RANKING (system enablers at top):
{influence_text}

TENSIONS (negative cross-impacts):
{chr(10).join(tension_pairs[:20]) if tension_pairs else "No strong tensions found. Consider mild resource competition between clusters."}

Propose exactly 6 scenario archetypes for horizon 2035. You MUST include:
- Exactly 2 EVOLUTIONARY scenarios (incremental progress, no breakthroughs dominate)
- Exactly 1 DISRUPTIVE scenario (one or more technology clusters achieve breakthrough, transforming the field)
- Exactly 1 CAUTIONARY scenario (promising technologies face deployment barriers, cost constraints, regulatory failure, organizational resistance, or unintended consequences — this is NOT just "stagnation" but active failure or negative outcomes)
- Exactly 1 WILDCARD scenario (low-probability event: geopolitical disruption, unexpected regulatory shift, supply chain collapse, or paradigm-breaking scientific discovery)
- Exactly 1 scenario of your choice (any type)

Rules:
- Every scenario must have at least 2 different states across clusters
- At least one scenario should have a cluster at "stagnation"
- The CAUTIONARY scenario should explore how even "breakthrough" technologies can fail to deliver value
- The WILDCARD scenario should explore a genuinely surprising development, not just "more technology"
- Justify each state assignment from the CIB data

For each archetype, provide a "perspective" — a 3-5 word framing phrase that captures the core narrative angle (e.g., "regulatory overreach stifles innovation", "quantum sensing reshapes enforcement", "cost barriers fragment adoption", "geopolitical tensions disrupt supply chains").

Return JSON:
{{
  "archetypes": [
    {{
      "short_name": "concise scenario name",
      "scenario_type": "evolutionary" or "disruptive" or "cautionary" or "wildcard",
      "perspective": "3-5 word framing phrase",
      "cluster_states": {{
        "1": "breakthrough/steady_progress/stagnation",
        "2": "breakthrough/steady_progress/stagnation",
        "3": "breakthrough/steady_progress/stagnation",
        "4": "breakthrough/steady_progress/stagnation"
      }},
      "rationale": "why these states, referencing CIB clusters and tensions"
    }}
  ]
}}"""

archetype_result = safe_chat_json(
    archetype_prompt,
    system="You are a technology foresight expert designing diverse scenario archetypes from cross-impact data. You value intellectual honesty and explore futures that are uncomfortable, not just optimistic.",
    temperature=0.4,
)

archetypes = archetype_result.get("archetypes", [])
print(f"\n=== LLM Proposed {len(archetypes)} Scenario Archetypes ===\n")

scenario_configs = []
for arch in archetypes:
    cluster_states = arch["cluster_states"]
    scenario_drivers = []
    driver_states = {}

    for cid_str, state in cluster_states.items():
        cid = int(cid_str)
        for idx in clusters.get(cid, []):
            d = cib_drivers_list[idx]
            if d:
                scenario_drivers.append(d)
                driver_states[d.id] = state

    stype_str = arch.get("scenario_type", "evolutionary")
    stype = ScenarioType(stype_str) if stype_str in [e.value for e in ScenarioType] else ScenarioType.EVOLUTIONARY

    # Within-cluster state variance for cautionary/wildcard
    if stype in (ScenarioType.CAUTIONARY, ScenarioType.WILDCARD):
        for cid_str, state in cluster_states.items():
            cid = int(cid_str)
            if state == "breakthrough" and len(clusters.get(cid, [])) > 1:
                cluster_driver_influence = [
                    (idx, cib["influence"].get(cib_driver_ids[idx], 0))
                    for idx in clusters[cid]
                ]
                cluster_driver_influence.sort(key=lambda x: x[1], reverse=True)
                for rank, (idx, _) in enumerate(cluster_driver_influence):
                    d = cib_drivers_list[idx]
                    if d and rank > 0:
                        driver_states[d.id] = "steady_progress"

    state_counts = {}
    for s in driver_states.values():
        state_counts[s] = state_counts.get(s, 0) + 1

    print(f"  [{stype.value:14s}] {arch['short_name']}")
    print(f"    Perspective: {arch.get('perspective', '')}")
    print(f"    States: {state_counts}")
    print(f"    Rationale: {arch.get('rationale', '')[:120]}...")
    print()

    scenario_configs.append({
        "name": arch["short_name"],
        "type": stype,
        "perspective": arch.get("perspective", ""),
        "drivers": scenario_drivers,
        "states": driver_states,
        "rationale": arch.get("rationale", ""),
        "cluster_states": cluster_states,
    })

print(f"Will generate {len(scenario_configs)} scenarios")

=== CIB-Based Driver Clusters (4 clusters) ===

Cluster 1 (avg influence: -5.0, avg dependence: +2.5):
  • Direction Finding Module and Space-Based Spectrum 
  • AI-Native 6G Networks with Embedded Spectrum Sensi

Cluster 2 (avg influence: +6.0, avg dependence: +3.0):
  • Real-Time Digitizer and Photonic Signal Processing
  • Panorama Scan Engine and Shift to Continuous, Auto
  • I/Q Data Streaming Interface and Edge Computing fo
  • Federated Learning and Energy-Efficient AI Archite
  • AI-Enabled Software-Defined Radio (SDR) and Open R

Cluster 3 (avg influence: +10.0, avg dependence: +6.5):
  • Ultra Wideband RF Frontend and Millimeter-Wave/Sub
  • Cross-Border and Harmonized Spectrum Monitoring Co

Cluster 4 (avg influence: +4.2, avg dependence: +5.6):
  • High Sensitivity Receiver and Quantum RF Sensing w
  • AI-Driven Spectrum Sensing, Prediction, and Dynami
  • Dynamic Spectrum Sharing and Real-Time Automated E
  • Increasing Cybersecurity Requirements for Monitori
  • Digital T


=== LLM Proposed 6 Scenario Archetypes ===

  [evolutionary  ] Incremental Integration
    Perspective: steady tech maturation
    States: {'steady_progress': 14}
    Rationale: All clusters progress incrementally without breakthroughs. Cluster 3 (highest influence) and Cluster 2 (high influence) ...

  [disruptive    ] Quantum-AI Synergy
    Perspective: quantum sensing reshapes enforcement
    States: {'steady_progress': 2, 'breakthrough': 12}
    Rationale: Clusters 2, 3, and 4 achieve breakthroughs, leveraging ultra wideband RF frontends, photonic processing, AI-driven dynam...

  [cautionary    ] Tech Promise, Policy Pitfalls
    Perspective: regulatory failure blocks value
    States: {'stagnation': 7, 'breakthrough': 2, 'steady_progress': 5}
    Rationale: Clusters 2 and 3 achieve breakthroughs (high influence enablers), but Clusters 1 and 4 stagnate due to regulatory failur...

  [wildcard      ] Spectrum Cold War
    Perspective: geopolitical tensions disrupt supply chains
  

## CIB Consistency Check

Validate driver-state combinations against the CIB matrix.
If Driver A = "breakthrough" and CIB[A→B] ≤ -1, then B cannot also be "breakthrough" — downgrade to "steady_progress".

In [3]:
def check_cib_consistency(driver_ids: list[str], states: list[str], threshold: int = -1) -> tuple[list[str], list[str]]:
    """Check and fix contradictions: if A='breakthrough' and CIB[A→B] <= threshold, B can't be 'breakthrough'."""
    adjusted = list(states)
    adjustments = []

    for i, (did_a, state_a) in enumerate(zip(driver_ids, adjusted)):
        if state_a != "breakthrough":
            continue
        idx_a = cib_id_to_idx.get(did_a)
        if idx_a is None:
            continue

        for j, (did_b, state_b) in enumerate(zip(driver_ids, adjusted)):
            if i == j or state_b != "breakthrough":
                continue
            idx_b = cib_id_to_idx.get(did_b)
            if idx_b is None:
                continue

            score = int(cib_matrix[idx_a][idx_b])
            if score <= threshold:
                adjusted[j] = "steady_progress"
                da = driver_by_id.get(did_a)
                db = driver_by_id.get(did_b)
                adjustments.append(
                    f"  CIB[{da.name[:30] if da else did_a} → {db.name[:30] if db else did_b}] = {score}: "
                    f"downgraded '{db.name[:30] if db else did_b}' to steady_progress"
                )

    return adjusted, adjustments


# apply consistency check to each config
print("=== CIB Consistency Check ===\n")
for config in scenario_configs:
    d_ids = [d.id for d in config["drivers"]]
    states = [config["states"][d.id] for d in config["drivers"]]

    adjusted_states, adj_log = check_cib_consistency(d_ids, states)

    if adj_log:
        print(f"[{config['name']}] — {len(adj_log)} adjustment(s):")
        for line in adj_log:
            print(line)
        config["adjusted_states"] = adjusted_states
    else:
        print(f"[{config['name']}] — consistent ✓")
        config["adjusted_states"] = states

=== CIB Consistency Check ===

[Incremental Integration] — consistent ✓
[Quantum-AI Synergy] — 3 adjustment(s):
  CIB[Real-Time Digitizer and Photon → High Sensitivity Receiver and ] = -1: downgraded 'High Sensitivity Receiver and ' to steady_progress
  CIB[Panorama Scan Engine and Shift → Real-Time Digitizer and Photon] = -1: downgraded 'Real-Time Digitizer and Photon' to steady_progress
  CIB[Panorama Scan Engine and Shift → I/Q Data Streaming Interface a] = -1: downgraded 'I/Q Data Streaming Interface a' to steady_progress
[Tech Promise, Policy Pitfalls] — consistent ✓
[Spectrum Cold War] — consistent ✓
[Fragmented Futures] — consistent ✓
[AI-Embedded 6G Leap] — 3 adjustment(s):
  CIB[Direction Finding Module and S → AI-Native 6G Networks with Emb] = -1: downgraded 'AI-Native 6G Networks with Emb' to steady_progress
  CIB[Direction Finding Module and S → High Sensitivity Receiver and ] = -1: downgraded 'High Sensitivity Receiver and ' to steady_progress
  CIB[Dynamic Spectrum Sharin

In [4]:
SYSTEM_PROMPTS = {
    "evolutionary": "You are a pragmatic technology analyst writing grounded future scenarios for spectrum monitoring. Focus on operational realities, incremental changes, and practical constraints.",
    "disruptive": "You are a technology foresight expert writing vivid future scenarios for spectrum monitoring. Explore both the promise and the disruption costs of breakthrough technologies.",
    "cautionary": "You are a critical technology analyst writing scenarios that examine what can go wrong with promising technologies. Focus on deployment barriers, organizational resistance, cost overruns, regulatory lag, and unintended consequences.",
    "wildcard": "You are a scenario planner exploring unlikely but plausible futures for spectrum monitoring. Focus on surprising developments, cascading effects, and second-order consequences.",
}

scenarios: list[Scenario] = []
generated_titles: list[str] = []
used_chunk_ids: set[str] = set()

for config in scenario_configs:
    print(f"\n{'='*60}")
    print(f"Generating: {config['name']} [{config['type'].value}]")
    print(f"Perspective: {config.get('perspective', '')}")
    print(f"{'='*60}")

    adjusted_states = config["adjusted_states"]

    assumptions = []
    for i, d in enumerate(config["drivers"]):
        state = adjusted_states[i]
        assumptions.append(DriverAssumption(
            driver_id=d.id,
            state=state,
            description=f"{d.name}: {state}",
        ))

    assumptions_text = "\n".join([
        f"- {a.description} (Driver origin: {next((d.origin.value for d in drivers if d.id == a.driver_id), '?')})"
        for a in assumptions
    ])

    # CIB context — include |score| >= 1 for richer context
    cib_context_parts = []
    scenario_driver_ids = [d.id for d in config["drivers"]]
    for did_a in scenario_driver_ids:
        idx_a = cib_id_to_idx.get(did_a)
        if idx_a is None:
            continue
        da = driver_by_id.get(did_a)
        for did_b in scenario_driver_ids:
            if did_a == did_b:
                continue
            idx_b = cib_id_to_idx.get(did_b)
            if idx_b is None:
                continue
            score = int(cib_matrix[idx_a][idx_b])
            if abs(score) >= 1:
                db = driver_by_id.get(did_b)
                if score >= 2:
                    effect = "strongly promotes"
                elif score == 1:
                    effect = "mildly promotes"
                elif score == -1:
                    effect = "mildly inhibits"
                else:
                    effect = "strongly inhibits"
                cib_context_parts.append(f"- {da.name[:40]} {effect} {db.name[:40]} (score: {score:+d})")
    cib_context = "\n".join(cib_context_parts) if cib_context_parts else "No notable cross-impacts among this scenario's drivers."

    existing_titles_block = ""
    if generated_titles:
        existing_titles_block = f"Previously generated scenario titles (yours must be clearly distinct):\n" + "\n".join(f"- {t}" for t in generated_titles)

    # State-aware RAG retrieval
    state_phrases = []
    for i, d in enumerate(config["drivers"]):
        state = adjusted_states[i]
        if state == "breakthrough":
            state_phrases.append(f"{d.name} breakthrough advances")
        elif state == "stagnation":
            state_phrases.append(f"{d.name} challenges barriers limitations")
        else:
            state_phrases.append(f"{d.name} incremental progress")

    perspective = config.get("perspective", "")
    query_text = f"{' '.join(state_phrases[:4])} {perspective} spectrum monitoring 2035"
    query_emb = embed([query_text[:500]])[0]
    rag = collection.query(query_embeddings=[query_emb], n_results=8, include=["documents", "metadatas"])

    # Prefer novel chunks not yet used by previous scenarios
    novel = [(rag["ids"][0][i], rag["documents"][0][i], rag["metadatas"][0][i])
             for i in range(len(rag["ids"][0])) if rag["ids"][0][i] not in used_chunk_ids]
    reused = [(rag["ids"][0][i], rag["documents"][0][i], rag["metadatas"][0][i])
              for i in range(len(rag["ids"][0])) if rag["ids"][0][i] in used_chunk_ids]
    ranked = (novel + reused)[:5]

    rag_text = "\n\n---\n\n".join([
        f"[Chunk ID: {cid}] (Source: {meta['source_title']})\n{doc}"
        for cid, doc, meta in ranked
    ])
    rag_chunk_ids = [cid for cid, _, _ in ranked]

    # Get type-specific narrative guide
    narrative_guide = prompts.SCENARIO_NARRATIVE_GUIDE.get(
        config["type"].value, prompts.SCENARIO_NARRATIVE_GUIDE["evolutionary"]
    )

    prompt = prompts.SCENARIO_GENERATE.format(
        driver_assumptions=assumptions_text,
        scenario_type=config["type"].value,
        perspective=perspective,
        narrative_guide=narrative_guide,
        existing_titles_block=existing_titles_block,
        cib_context=cib_context,
        rag_chunks=rag_text,
    )

    system = SYSTEM_PROMPTS.get(config["type"].value, SYSTEM_PROMPTS["evolutionary"])
    result = safe_chat_json(prompt, system=system, temperature=0.7)

    # merge source IDs — only keep IDs that exist in the RAG results (no driver phantom IDs)
    all_source_ids = list(rag_chunk_ids)
    all_source_ids.extend(result.get("source_chunk_ids_used", []))
    all_source_ids = list(set(all_source_ids))

    used_chunk_ids.update(rag_chunk_ids)

    scenario = Scenario(
        title=result.get("title", config["name"]),
        narrative=result.get("narrative", ""),
        type=config["type"],
        perspective=perspective,
        key_tensions=result.get("key_tensions", []),
        assumptions=assumptions,
        source_chunk_ids=all_source_ids,
    )
    scenarios.append(scenario)
    generated_titles.append(scenario.title)

    print(f"\nTitle: {scenario.title}")
    print(f"Narrative preview: {scenario.narrative[:300]}...")
    print(f"Key tensions: {scenario.key_tensions}")
    print(f"Key changes: {result.get('key_changes', [])}")
    print(f"Source chunks: {len(scenario.source_chunk_ids)}")

# diversity check
print(f"\n=== Generated {len(scenarios)} scenarios ===")
print(f"\nDiversity check:")
for s in scenarios:
    state_counts = {}
    for a in s.assumptions:
        state_counts[a.state] = state_counts.get(a.state, 0) + 1
    has_mixed = len(state_counts) >= 2
    print(f"  {'✓' if has_mixed else '✗'} [{s.type.value:14s}] {s.title[:50]} — {state_counts}")

# chunk overlap check (RAG-only, no driver phantoms)
print(f"\nChunk overlap:")
all_chunks = [set(s.source_chunk_ids) for s in scenarios]
for i in range(len(all_chunks)):
    for j in range(i+1, len(all_chunks)):
        overlap = len(all_chunks[i] & all_chunks[j])
        total = len(all_chunks[i] | all_chunks[j])
        pct = overlap/total*100 if total else 0
        flag = " ⚠" if pct > 50 else ""
        print(f"  S{i+1} vs S{j+1}: {pct:.0f}%{flag}")


Generating: Incremental Integration [evolutionary]
Perspective: steady tech maturation



Title: Incremental Integration and Persistent Complexity in 2035 Spectrum Monitoring
Narrative preview: SETTING:
In 2035, a spectrum monitoring operator begins the day at a national regulatory agency’s operations center, overseeing a distributed and layered monitoring infrastructure. The operator’s workstation integrates data streams from ground-based AI-enabled software-defined radios (SDRs), edge co...
Key tensions: ['Mild incompatibilities and tradeoffs between space-based direction finding modules and AI-native 6G embedded spectrum sensing data requiring human oversight', 'Balancing advanced cybersecurity requirements against system flexibility and rapid innovation adoption', 'Training and operational challenges posed by increasingly complex, multi-modal AI-driven monitoring environments']
Key changes: ['Continuous, real-time spectrum monitoring enabled by AI-driven dynamic sensing and space-based LEO satellite constellations with direction finding modules', 'Integration of federa


Title: Quantum Leap: Enforcing Spectrum Sovereignty through Atomic Precision in 2035
Narrative preview: SETTING: By 2035, the integration of quantum RF sensing based on Rydberg atoms marks a seismic shift in spectrum monitoring enforcement. After years of steady progress in quantum sensor miniaturization and room-temperature operation, regulators worldwide deploy these sensors within distributed, AI-e...
Key tensions: ['Balancing AI-driven automated enforcement decisions with transparency and regulatory accountability', 'Reconciling national sovereignty concerns with the need for cross-border federated learning and data sharing', 'Managing workforce transition challenges amid rapid obsolescence of classical spectrum monitoring systems']
Key changes: ['Deployment of Rydberg atom-based quantum RF sensors achieving 100x sensitivity improvement over classical receivers', 'Integration of continuous, automated real-time monitoring via panorama scan engines with AI-driven enforcement', 'Fede


Title: Stalled Spectrum Oversight: How Regulatory Gridlock Undermined 2035 Monitoring Advances
Narrative preview: SETTING: By 2035, spectrum regulators worldwide face an unprecedented crisis. While breakthrough hardware innovations, notably in real-time digitizers with photonic signal processing and ultra-wideband RF frontends extending into millimeter-wave and sub-THz bands, promised a revolution in spectrum m...
Key tensions: ['Tradeoff between investing in breakthrough hardware versus maturing AI-native sensing and sharing protocols', 'Balancing cybersecurity requirements with innovation and deployment speed', 'Reconciling national regulatory sovereignty with the need for harmonized cross-border spectrum monitoring']
Key changes: ['Breakthrough hardware innovations (photonic signal processing, ultra-wideband frontends) deployed but underutilized due to regulatory lag', 'Stagnation of AI-native 6G embedded sensing and cognitive spectrum access technologies prevents full system integ


Title: Spectrum Monitoring in Fragmented Futures: Supply Chain Shocks and Quantum Reliance by 2035
Narrative preview: THE TRIGGER: In the early 2030s, escalating geopolitical tensions among major powers triggered unprecedented disruptions in global supply chains, especially those critical to advanced electronics and photonic components essential for spectrum monitoring hardware. Trade embargoes and national securit...
Key tensions: ['Tradeoff between coverage completeness and sovereignty/security in fragmented, localized monitoring networks.', 'Balancing the adoption of quantum sensing’s novel capabilities against integration challenges with legacy spectrum infrastructure.', 'Managing cybersecurity in isolated networks versus benefits of collaborative federated learning approaches.']
Key changes: ['Quantum RF sensing matures as the primary technology for sensitive, wideband ground-based spectrum monitoring, circumventing fragile supply chains.', 'Space-based LEO satellite spectrum mon


Title: Fragmented Realities: Cost-Driven Divergence in 2035 Spectrum Monitoring
Narrative preview: It is a typical morning in 2035 at a mid-sized national spectrum regulatory agency. The operator logs into a spectrum monitoring dashboard that reflects a patchwork of technologies across the agency’s coverage area. Incremental advances in real-time digitizers, photonic signal processing, and AI-dri...
Key tensions: ['Balancing investment in cutting-edge AI-enabled SDR and ORAN architectures against slow procurement cycles and budget constraints.', 'Reconciling heterogeneous data from legacy and modern monitoring equipment to maintain reliable enforcement.', 'Managing cybersecurity risks amid expanding networked monitoring infrastructure with uneven technical expertise.']
Key changes: ['Incremental improvements in real-time digitizers, photonic processing, and panorama scan engines enable near-continuous automated monitoring in well-funded regions.', 'Federated learning and edge computin


Title: AI-Native 6G: The Real-Time Spectrum Sentinels of 2035
Narrative preview: SETTING:
By 2035, the integration of AI-native 6G networks has catalyzed a paradigm shift in spectrum monitoring. Unlike previous generations where spectrum sensing was an external, often reactive process, 6G embeds AI-driven spectrum sensing directly into the network fabric. This breakthrough is po...
Key tensions: ['Balancing space-based and terrestrial monitoring amid mild cross-inhibitions between direction-finding modules and AI-native 6G components', 'Institutional resistance to AI-driven autonomous monitoring versus the need for rapid retraining and digital transformation', 'Geopolitical fragmentation impeding cross-border spectrum monitoring harmonization despite technological capabilities']
Key changes: ['Spectrum sensing fully embedded and automated within AI-native 6G network infrastructure', 'Real-time dynamic spectrum sharing enabled by federated learning and edge computing', 'Digital twin ne

In [5]:
# save scenarios
scenario_state = {
    "scenarios": [s.model_dump(mode="json") for s in scenarios],
}
with open("../data/outputs/scenario_state.json", "w") as f:
    json.dump(scenario_state, f, indent=2)
print(f"Saved {len(scenarios)} scenarios")

Saved 6 scenarios
